# Mixed-effects models for valence × depression interactions

This notebook asks a more targeted question than the previous one: does
depression severity moderate how much attention goes to one valence versus the other, within each pair type? For each scene-level outcome (dwell time, fixation proportion, revisit count, plus the 500-ms early-attention window), we fit one linear mixed-effects model per pair with the two-level valence factor:

`y ~ score * C(valence, Treatment(reference=<ref>)) + trial_norm + (1 + trial_norm | uid)`

Fitting one model per pair keeps each coefficient tied to a single,
interpretable contrast:

- `neg_vs_pos` — reference = positive, target = negative. Coefficient
  directly tests the classic depression attentional-bias hypothesis:
  do depressed participants shift attention toward negative and away from
  positive, when both are on screen?
- `neg_vs_neu` — reference = neutral, target = negative. Tests
  vigilance to negative versus a neutral baseline - a weaker variant of
  the threat-bias literature not involving explicit reward competition.
- `pos_vs_neu` — reference = neutral, target = positive. Tests the
  protective-bias hypothesis, that healthy participants preferentially attend to positive stimuli over neutral, and that depression attenuates this effect.

`trial_norm` appears as a main effect but not as a three-way interaction
with score × valence. This treats within-session drift as a nuisance
covariate rather than a second hypothesis to test, which fits what this
notebook is trying to do.

In [0]:
%pip install statsmodels -q

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd

from src.features.session_aggregation import PAIRS
from src.evaluation.lmm_valence import fit_all_per_pair, apply_fdr, plot_pair_valence_effect

## 1. Load data

In [0]:
OUTCOMES = {
    "dwell_time_ms": [
        "dwell_time_ms_negative", "dwell_time_ms_positive", "dwell_time_ms_neutral",
    ],
    "dwell_time_500ms": [
        "dwell_time_500ms_negative", "dwell_time_500ms_positive", "dwell_time_500ms_neutral",
    ],
    "fixation_proportion": [
        "fixation_proportion_negative", "fixation_proportion_positive", "fixation_proportion_neutral",
    ],
    "revisit_count": [
        "revisit_count_negative", "revisit_count_positive", "revisit_count_neutral",
    ],
}

VALID_PAIRS = ["negative_vs_positive", "negative_vs_neutral", "neutral_vs_positive"]
DURATION_MIN_MS = 500
DURATION_MAX_MS = 10000

scene_metrics = spark.table("anima.scene_metrics")
df_forms = spark.table("anima.forms")

df_stimulus = (scene_metrics
    .filter(F.col("scene_type") == "stimulus")
    .filter(F.col("scene_quality_valid") == True)
    .filter(F.col("scene_valence_pair").isin(VALID_PAIRS))
    .filter(F.col("duration_ms").between(DURATION_MIN_MS, DURATION_MAX_MS)))

w = Window.partitionBy("session_id").orderBy("scene_index")
numbered = df_stimulus.withColumn("trial_num", F.row_number().over(w))

all_value_cols = [c for cols in OUTCOMES.values() for c in cols]
cols = ["session_id", "scene_index", "trial_num", "scene_valence_pair"] + all_value_cols

df_wide = numbered.select(*cols).join(
    df_forms.select("session_id", "uid", "phq9_score", "bdi_score"),
    on="session_id", how="inner",
).toPandas()

df_wide["trial_norm"] = df_wide.groupby("session_id")["trial_num"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else 0
)
df_wide["phq9_z"] = (df_wide["phq9_score"] - df_wide["phq9_score"].mean()) / df_wide["phq9_score"].std()
df_wide["bdi_z"] = (df_wide["bdi_score"] - df_wide["bdi_score"].mean()) / df_wide["bdi_score"].std()

print(f"Rows: {len(df_wide)}, users: {df_wide['uid'].nunique()}")
print(f"Outcomes × pairs = {len(OUTCOMES)} × {len(PAIRS)} = {len(OUTCOMES) * len(PAIRS)} models per score")

In [0]:
ID_VARS = ["session_id", "scene_index", "trial_num", "trial_norm", "uid", "phq9_score", "bdi_score", "phq9_z", "bdi_z", "scene_valence_pair"]

## 2. PHQ-9

### 2.1 Fit all models

In [0]:
phq_results, phq_summary = fit_all_per_pair(df_wide, OUTCOMES, "phq9_z", PAIRS, ID_VARS)
phq_summary = apply_fdr(phq_summary)
phq_summary[["outcome", "pair_suffix", "reference", "focal", "interaction_coef", "interaction_pval_fdr", "interaction_d", "icc", "n_obs"]].sort_values("interaction_pval_fdr")

### 2.2 FDR-significant interactions (α = 0.05)

In [0]:
phq_summary[phq_summary["interaction_pval_fdr"] < 0.05][["outcome", "pair_suffix", "reference", "focal", "interaction_coef", "interaction_pval_fdr", "interaction_d"]].sort_values("interaction_pval_fdr")

### 2.3 Dwell time by valence — one plot per pair

In [0]:
from src.evaluation.lmm_valence import melt_one_pair

for pair_name, pair_suffix, valences in PAIRS:
    sub = df_wide[df_wide["scene_valence_pair"] == pair_name]
    df_long = melt_one_pair(sub, OUTCOMES["dwell_time_ms"], valences, ID_VARS)
    plot_pair_valence_effect(df_long, "phq9_score", "PHQ-9", pair_suffix, y_label="Dwell time (ms)")

## 3. BDI-II

### 3.1 Fit all models

In [0]:
bdi_results, bdi_summary = fit_all_per_pair(df_wide, OUTCOMES, "bdi_z", PAIRS, ID_VARS)
bdi_summary = apply_fdr(bdi_summary)
bdi_summary[["outcome", "pair_suffix", "reference", "focal", "interaction_coef", "interaction_pval_fdr", "interaction_d", "icc", "n_obs"]].sort_values("interaction_pval_fdr")

### 3.2 FDR-significant interactions (α = 0.05)

In [0]:
bdi_summary[bdi_summary["interaction_pval_fdr"] < 0.05][["outcome", "pair_suffix", "reference", "focal", "interaction_coef", "interaction_pval_fdr", "interaction_d"]].sort_values("interaction_pval_fdr")

### 3.3 Dwell time by valence — one plot per pair

In [0]:
for pair_name, pair_suffix, valences in PAIRS:
    sub = df_wide[df_wide["scene_valence_pair"] == pair_name]
    df_long = melt_one_pair(sub, OUTCOMES["dwell_time_ms"], valences, ID_VARS)
    plot_pair_valence_effect(df_long, "bdi_score", "BDI-II", pair_suffix, y_label="Dwell time (ms)")

## 4. Detailed fit: dwell time on neg_vs_pos

In [0]:
key = ("dwell_time_ms", "neg_vs_pos")
if key in phq_results:
    print(phq_results[key].summary())
    print()
    print(bdi_results[key].summary())
else:
    print(f"No fit available for {key}")

## 5. PHQ-9 vs BDI-II comparison

In [0]:
comparison = phq_summary[["outcome", "pair_suffix", "interaction_coef", "interaction_pval_fdr", "interaction_d", "icc"]].merge(
    bdi_summary[["outcome", "pair_suffix", "interaction_coef", "interaction_pval_fdr", "interaction_d"]],
    on=["outcome", "pair_suffix"], suffixes=("_phq", "_bdi"),
).sort_values("interaction_pval_fdr_phq")
comparison

In [0]:
sig_phq = comparison["interaction_pval_fdr_phq"] < 0.05
sig_bdi = comparison["interaction_pval_fdr_bdi"] < 0.05

pd.Series({
    "both": (sig_phq & sig_bdi).sum(),
    "phq_only": (sig_phq & ~sig_bdi).sum(),
    "bdi_only": (~sig_phq & sig_bdi).sum(),
    "neither": (~sig_phq & ~sig_bdi).sum(),
    "n_tests": len(comparison),
})

## 6. Reliability (ICC)

In [0]:
phq_summary[["outcome", "pair_suffix", "icc"]].sort_values("icc", ascending=False)

In [0]:
phq_summary["icc"].agg(["mean", "median"]).round(3)